# Foundation Models: Time series with different lengths and different exogenous variables

Real multi-series datasets are rarely tidy. Some series start recording later than others, some stop before the end of the observation period, a sensor fails and leaves a block of missing values, and not every series has the same set of covariates available. When these series are modelled with a global machine learning model, skforecast handles this heterogeneity in [ForecasterRecursiveMultiSeries](../user_guides/multi-series-with-different-length-and-different_exog.ipynb) by building one training matrix from whatever each series has.

Foundation models pose a different challenge. They do not train on the data: at predict time the whole batch of series is handed to a pre-trained backend (Chronos-2, TimesFM, TabPFN-TS, and so on), and each backend has its own rules about what a batch may contain. This guide explains where those rules come from, how [ForecasterFoundation](../api/ForecasterFoundation.md) and [FoundationModel](../api/FoundationModel.md) deal with them so that the user does not have to, and what each adapter requires and tolerates.

<div class="admonition note" name="html-admonition" style="background: rgba(0,191,191,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #00bfa5; border-color: #00bfa5; padding-left: 10px; padding-right: 10px;">

<p class="title">
    <i style="font-size: 18px; color:#00bfa5;"></i>
    <b style="color: #00bfa5;">&#128161; Tip</b>
</p>

This guide assumes familiarity with the basics of <a href="../user_guides/foundation-forecasting-models.html">forecasting with foundation models</a> in skforecast: the role of the context window, the <code>fit</code> / <code>predict</code> semantics and the supported backends.

</div>

## The challenge

A multi-series dataset is heterogeneous when its series differ along one or more of these axes:

| Axis | Example |
|:-----|:--------|
| **Length and time span** | Series `A` covers the whole year, series `B` starts in July, series `C` stops in June. |
| **Exogenous variables** | `A` has `temperature` and `holiday`, `B` has only `holiday`, `C` has none. |
| **Missing values** | `A` has a two-week gap of NaN in the middle of the year, or trailing NaN at the end of the training span. |

For a foundation model, these differences matter because inference is done in **batches**: a single call to the backend receives every series (its context window and its covariates) and returns every forecast. The backends impose constraints on that batch that a user working series by series would never notice:

- **Identical covariate columns per batch.** Chronos-2 and TS-ICL validate that every element of the batch carries exactly the same `past_covariates` and `future_covariates` keys and reject the call otherwise (`Heterogeneous lists are not supported`). TimesFM 3.0 stacks the covariate arrays of all series into one tensor, so every series must have the same number of covariate columns. TabICL builds one long-format frame with a single set of columns for all series.

- **Padding a missing column with NaN is not neutral.** A tempting workaround is to give every series the union of all columns and fill the missing ones with NaN. For some backends this is exactly what the library expects (TFC-T0 defines NaN as "covariate absent", TabPFN-TS imputes missing cells). For Chronos-2 it silently changes the forecast: in tests with `chronos-2-small`, adding two all-NaN covariate columns to a series moved its forecast by about 3 to 30 percent of its scale, whereas forecasting the same series alone or together with series that share its columns changed it by less than 0.001 percent.

- **Different lengths inside a batch.** Backends left-pad shorter series to the length of the longest one. This is harmless in most cases, but TimesFM 3.0 pads the covariates too, and with covariates present the padded values enter the model. In tests with `timesfm-3.0`, a 30-observation series forecast next to a 91-observation series moved by about 2 percent of its scale when covariates were present and by less than 0.001 percent without them.

- **NaN in the target.** Most backends treat NaN as missing (Chronos-2), interpolate it (TimesFM) or drop those rows (TabICL, TabPFN-TS). Nori's regressor rejects NaN outright, and no backend can do anything with a context window that is entirely NaN.

- **Backtesting multiplies the problem.** A series that ends before the end of the span, or has a block of NaN inside a fold, would be predicted at the wrong dates or with a context and exog that no longer line up, and a series with no data in a fold's test window has nothing to be evaluated against.

## How skforecast solves it

The design principle is: **one contract in `FoundationModel`, thin adapters**. All the reasoning about which series can share a backend call, how the exog must be aligned and when a series can be predicted lives in `FoundationModel.predict` and in `backtesting_foundation`. Adapters only translate already-normalized per-series inputs into a backend call, and they declare their constraints through four read-only attributes:

| Attribute | Meaning |
|:----------|:--------|
| `allow_exog` | The backend uses exogenous variables at all. If `False`, `exog` and `context_exog` are ignored with an `IgnoredArgumentWarning`. |
| `supports_past_only_covariates` | Historical exog columns that have no future values are used as past-only covariates. If `False`, those columns are ignored with an `IgnoredArgumentWarning`. |
| `supports_heterogeneous_covariates` | The backend accepts, in one call, series whose exog columns differ. If `False`, `FoundationModel` groups the series by their exog columns and calls the adapter once per group. |
| `supports_nan_in_series` | The backend accepts NaN in the context. If `False`, a context with NaN raises a `ValueError` before the adapter is called. |

The attributes are exposed on `FoundationModel` and `ForecasterFoundation`, so they can be inspected before deciding how to prepare the data:

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

In [ ]:
from skforecast.foundation import FoundationModel, ForecasterFoundation

forecaster = ForecasterFoundation(
    estimator=FoundationModel(model_id="autogluon/chronos-2-small")
)
forecaster.allow_exog                          # True
forecaster.supports_past_only_covariates       # True
forecaster.supports_heterogeneous_covariates   # False: series are grouped by exog columns
forecaster.supports_nan_in_series              # True

AttributeError: 'ForecasterFoundation' object has no attribute 'allow_exog'

### What `FoundationModel.predict` guarantees to every adapter

Whatever the input format (a dict of series, a wide DataFrame, a long-format DataFrame) and whichever path the call comes from (`predict`, `predict_interval`, `predict_quantiles` or backtesting), the adapter always receives inputs that satisfy four invariants:

1. **`context_exog[name]` shares the index of `context[name]`.** The historical exog of each series is reindexed to its context window. Rows outside the window are dropped; context timestamps missing from the exog are added as NaN and reported once with a `MissingValuesWarning`.

2. **`exog[name]` has exactly `steps` rows aligned to the forecast horizon, or is `None`.** The future exog is reindexed to the horizon of each series (which starts right after its own context, so series ending on different dates get different horizons). Missing timestamps are filled with NaN and reported with a `MissingValuesWarning`.

3. **Every series in one adapter call has the same covariate signature** when `supports_heterogeneous_covariates` is `False`. The signature of a series is the pair `(past-only columns, future columns)`, both sorted. Series with the same signature are batched together, series with a different signature go in separate calls, and the merged output keeps the order of the input series.

4. **No context contains NaN** when `supports_nan_in_series` is `False`.

Everything the central layer does is **splitting and aligning**. It never imputes, drops or fabricates values or columns: a series only ever carries its own exog columns, and a NaN that reaches the backend is either a NaN present in the user's data or a timestamp the user did not provide. Anything that changes values is a documented, backend-specific decision inside the adapter (see the adapter table below).

### Per-series validation of the exog columns

Before predicting, the columns of the future `exog` are compared, series by series, with the historical exog used as context (the one passed to `fit`, or `context_exog` when `context` is given):

| Situation for one series | Result |
|:-------------------------|:-------|
| A future column has no historical values | `ValueError`. No backend can use a covariate without its history. |
| A historical column has no future values | Past-only covariate. Used as such if `supports_past_only_covariates` is `True`, otherwise ignored with an `IgnoredArgumentWarning`. |
| Same columns in both | Known-future covariate. |
| No exog at all for this series | The series is forecast without covariates, even if other series have them. |

Column names are compared as sets, so the order does not matter. Series can have different subsets of columns and are validated independently.

### Grouping in practice

Consider four series forecast with Chronos-2 (`supports_heterogeneous_covariates=False`):

| Series | Historical exog | Future exog | Signature `(past-only, future)` |
|:-------|:----------------|:------------|:--------------------------------|
| `full`  | `p`, `k` | `k` | `(("p",), ("k",))` |
| `full2` | `k`, `p` | `k` | `(("p",), ("k",))` |
| `past`  | `p`      | none | `(("p",), ())` |
| `none`  | none     | none | `((), ())` |

`FoundationModel.predict` makes three backend calls: one with `full` and `full2`, one with `past` and one with `none`. Each series is forecast with its own covariates only, and the forecast of `none` is identical to the one obtained by predicting it alone. For an adapter with `supports_heterogeneous_covariates=True` (TFC-T0, TabPFN-TS, Nori, and the adapters that ignore exog) the four series go in a single call.

<div class="admonition note" name="html-admonition" style="background: rgba(0,184,212,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #00b8d4; border-color: #00b8d4; padding-left: 10px; padding-right: 10px;">

<p class="title">
    <i style="font-size: 18px; color:#00b8d4;"></i>
    <b style="color: #00b8d4;">&#9999;&#65039; Note</b>
</p>

The number of backend calls equals the number of distinct signatures, which is the minimum the backends allow. A dataset where every series has the same exog columns still makes a single call, exactly as before. For Chronos-2, <code>cross_learning=True</code> shares information among the series of the same group, since the backend cannot see series from other groups.

</div>

## Requirements common to every foundation model

The heterogeneity that skforecast absorbs is in lengths, time spans, exog columns and missing values. A few requirements still apply to every series in the batch, as they do for the rest of the library:

- All series must have the same **index type** (`DatetimeIndex` or `RangeIndex`) and the same **frequency**. Set it with `asfreq()` before fitting.
- A series cannot be **entirely NaN**.
- The exog of a series must share the index type of the series. Rows are matched by timestamp, so an exog can start or end on different dates than its series.
- Only the last `context_length` observations of each series are used. A series shorter than `context_length` is used in full; a series longer is truncated from the left.
- Covariates must be numeric (or boolean) for most backends. Chronos-2 and TS-ICL accept string and categorical columns natively; for the other backends, encode categoricals as numbers with `transformer_exog` or beforehand.

## Adapter requirements and limitations

The table summarizes, for each adapter, the declared capabilities and how the backend treats missing values. "Grouped" means `supports_heterogeneous_covariates=False`: `FoundationModel` calls the backend once per group of series sharing the same exog columns.

| Adapter (`model_id` prefix) | Exog | Past-only covariates | Batch with different exog columns | NaN in target | NaN in covariates |
|:----------------------------|:-----|:---------------------|:----------------------------------|:--------------|:------------------|
| **Chronos-2** (`autogluon/chronos`) | Past and future | Yes | Grouped | Treated as missing values | Treated as missing values |
| **TimesFM 2.5** (`google/timesfm-2.5`) | Ignored | No | Single call (no covariates) | Linearly interpolated, leading NaN trimmed | Not applicable |
| **TimesFM 3.0** (`google/timesfm-3.0`) | Past and known-future, numeric only | Yes | Grouped | Linearly interpolated, leading NaN trimmed | Linearly interpolated |
| **Moirai-2** (`Salesforce/moirai`) | Ignored | No | Single call (no covariates) | Accepted | Not applicable |
| **TabICL** (`soda-inria/tabicl`) | Known-future | No (ignored with a warning) | Grouped | Rows with NaN target dropped by the library | Accepted with a warning from the library |
| **TabPFN-TS** (`priorlabs/tabpfn`) | Known-future | No (ignored with a warning) | Single call | Rows with NaN target dropped by the library | Missing cells imputed by the library |
| **TFC-T0** (`theforecastingcompany/t0`) | Known-future | No (ignored with a warning) | Single call | Accepted | NaN means "covariate absent" |
| **Synthefy Nori** (`Synthefy/Nori`) | Known-future, numeric only | No (ignored with a warning) | Single call (one in-context fit per series) | Rows with NaN target or NaN feature dropped by the adapter | Rows dropped from the context |
| **TS-ICL** (`taharnbl/TS-ICL`) | Past and future | Yes | Grouped | Accepted | Accepted |

Details worth knowing per adapter:

**Chronos-2.** The backend validates that every element of a batch has the same covariate keys and agrees on whether each future covariate is available, so grouping is mandatory. Series of different lengths in the same group are fine. `cross_learning` applies within a group. Numeric and boolean columns are cast to `float32`; string and categorical columns are forwarded as-is.

**TimesFM 3.0.** `predict_batch` stacks the covariate arrays of the batch, so every series in a call must have the same number of past-only and known-future columns: grouping is mandatory. Series of different lengths in the same group are left-padded by the backend; with covariates present, the padding enters the model and the forecast of a short series can differ slightly (about 2 percent of its scale in tests) from the forecast obtained when it is predicted alone. Without covariates the effect is negligible. NaN inside the target and inside the covariates is linearly interpolated by the backend, and leading NaN in the target trims the context and its covariates accordingly. Covariates must be numeric.

**TimesFM 2.5 and Moirai-2.** They do not use covariates, so any exog is ignored with an `IgnoredArgumentWarning` and every series goes in a single call. Mixed lengths and NaN are handled by the backend.

**TabICL.** The backend builds one long-format frame for all series and uses only the covariate columns present in both the context and the future data, so grouping keeps a series from receiving NaN-filled columns it does not have. Rows whose target is NaN are dropped by the library. NaN in the future covariates is accepted but the library warns that it may affect quality. Historical columns without future values are ignored.

**TabPFN-TS.** The library handles a long-format frame where different series have different columns by imputing the missing cells, so all series go in a single call. Rows with NaN target are dropped by the library. Historical columns without future values are ignored.

**TFC-T0.** T0 defines NaN as an absent covariate value; the adapter pools the columns of all series and fills the missing cells with NaN, so a single call serves every series. Historical columns without future values are ignored.

**Synthefy Nori.** Each series is fitted in-context and predicted in its own `NoriRegressor` call, so there is no batching constraint. `NoriRegressor` rejects NaN, and the adapter drops the context rows whose target or covariates are NaN before the in-context fit. The running-index feature is an absolute offset, so dropping interior rows keeps the remaining rows correctly positioned in time. A series whose context has no row free of NaN raises a `ValueError`. Covariates must be numeric. A future exog with gaps is filled with NaN by `FoundationModel` and may be rejected by the regressor at predict time.

**TS-ICL.** As with Chronos-2, every element of a batch must carry identical covariate keys, so grouping is mandatory. Past-only covariates are supported.

<div class="admonition note" name="html-admonition" style="background: rgba(255,145,0,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #ff9100; border-color: #ff9100; padding-left: 10px; padding-right: 10px;">

<p class="title">
    <i style="font-size: 18px; color:#ff9100;"></i>
    <b style="color: #ff9100;">&#9888;&#65039; Warning</b>
</p>

The adapters are internal classes. Calling <code>adapter.predict</code> directly bypasses the alignment and grouping described here, and the backend error messages (for example Chronos-2's <code>Heterogeneous lists are not supported</code>) reappear. Always go through <code>ForecasterFoundation</code> or <code>FoundationModel</code>.

</div>

## Example: fit, predict and backtest heterogeneous series

The example uses the same dataset as the [global models guide](../user_guides/multi-series-with-different-length-and-different_exog.ipynb) on series of different lengths: five daily series in long format, together with four exogenous variables. Two series are deliberately stripped of some exogenous columns so that every axis of heterogeneity is present at once.

In [ ]:
# Libraries
# ==============================================================================
import pandas as pd
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.preprocessing import reshape_series_long_to_dict, reshape_exog_long_to_dict
from skforecast.model_selection import TimeSeriesFold, backtesting_foundation

# Load time series of multiple lengths and exogenous variables
# ==============================================================================
series = pd.read_csv(
    'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/demo_multi_series.csv'
)
exog = pd.read_csv(
    'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/demo_multi_series_exog.csv'
)
series['timestamp'] = pd.to_datetime(series['timestamp'])
exog['timestamp'] = pd.to_datetime(exog['timestamp'])

# Transform series and exog to dictionaries
# ==============================================================================
series_dict = reshape_series_long_to_dict(
    data      = series,
    series_id = 'series_id',
    index     = 'timestamp',
    values    = 'value',
    freq      = 'D'
)
exog_dict = reshape_exog_long_to_dict(
    data      = exog,
    series_id = 'series_id',
    index     = 'timestamp',
    freq      = 'D'
)

# Drop some exogenous variables for series 'id_1000' and 'id_1003'
# ==============================================================================
exog_dict['id_1000'] = exog_dict['id_1000'].drop(columns=['air_temperature', 'wind_speed'])
exog_dict['id_1003'] = exog_dict['id_1003'].drop(columns=['cos_day_of_week'])

# Partition data in train and test
# ==============================================================================
end_train = '2016-07-31 23:59:00'
series_dict_train = {k: v.loc[:end_train] for k, v in series_dict.items()}
exog_dict_train   = {k: v.loc[:end_train] for k, v in exog_dict.items()}
exog_dict_test    = {k: v.loc[end_train:] for k, v in exog_dict.items()}

fter reshaping, `reshape_series_long_to_dict` warns that `id_1003` is incomplete: setting the daily frequency introduces NaN where the raw data has no rows. The resulting dataset looks like this:

| Series | Span | Train length | NaN in target | Exogenous columns |
|:-------|:-----|-------------:|:--------------|:------------------|
| `id_1000` | 2016-01-01 to 2016-12-31 | 213 | No | `sin_day_of_week`, `cos_day_of_week` |
| `id_1001` | 2016-07-02 to 2016-12-31 | 30 | No | `sin_day_of_week`, `cos_day_of_week`, `air_temperature`, `wind_speed` |
| `id_1002` | 2016-01-01 to 2016-07-01 | 183 | No | `sin_day_of_week`, `cos_day_of_week`, `air_temperature`, `wind_speed` |
| `id_1003` | 2016-01-01 to 2016-12-31 | 213 | Yes, several blocks | `sin_day_of_week`, `air_temperature`, `wind_speed` |
| `id_1004` | 2016-05-02 to 2016-08-31 | 91 | No | `sin_day_of_week`, `cos_day_of_week`, `air_temperature`, `wind_speed` |

Mixed lengths (30 to 213 training observations), a series that starts late (`id_1001`), a series that ends before the split (`id_1002`, no test data at all), a series that ends one month into the test period (`id_1004`), three different exog subsets, and NaN blocks in `id_1003`.


In [ ]:
# Fit and predict
# ==============================================================================
# Available model_ids
# "autogluon/chronos-2-small",
# "google/timesfm-3.0-pytorch",
# "google/timesfm-2.5-200m-pytorch",
# "soda-inria/tabicl",
# "priorlabs/tabpfn-ts",
# "theforecastingcompany/t0-alpha",
# "Synthefy/Nori",
# "taharnbl/TS-ICL"

forecaster = ForecasterFoundation(
    estimator=FoundationModel(
        model_id       = "autogluon/chronos-2-small",
        context_length = 500,
    )
)
forecaster.fit(series=series_dict_train, exog=exog_dict_train)
predictions = forecaster.predict(steps=5, exog=exog_dict_test)
predictions.head(10)

What happens inside that `predict` call:

- The horizon of every series starts right after its own last training observation: `id_1000`, `id_1001`, `id_1003` and `id_1004` are forecast from 2016-08-01, `id_1002` from 2016-07-02.
- `exog_dict_test['id_1002']` is empty because that series (and its exog) ended before the split. Its future exog is reindexed to the horizon and filled with NaN, and a `MissingValuesWarning` names the series. Chronos-2 treats those NaN as missing covariate values.
- Every series is validated against its own history, so `id_1000` is forecast with two covariates, `id_1003` with three and the other series with four. Chronos-2 requires identical covariate keys in a batch, so the five series are forecast in three backend calls: one for `id_1000`, one for `id_1003`, and one for `id_1001`, `id_1002` and `id_1004`. With TabPFN-TS or TFC-T0 the same input would be a single call.
- The NaN blocks in `id_1003` stay in its context window. With a backend declaring `supports_nan_in_series=False`, `predict` would raise a `ValueError` naming the series.

The same code runs unchanged with any other covariate-aware `model_id` (`google/timesfm-3.0-pytorch`, `soda-inria/tabicl`, `priorlabs/tabpfn-ts`, `theforecastingcompany/t0-alpha`, `Synthefy/Nori`, `taharnbl/TS-ICL`); with `google/timesfm-2.5-200m-pytorch` or `Salesforce/moirai-2.0-R-small` the exog is ignored with an `IgnoredArgumentWarning`.

### Backtesting

`backtesting_foundation` applies the same alignment and grouping in every fold, and adds its own rules so that every prediction is dated inside its fold and can be evaluated:

- The **context** of a series in a fold is the series from its first observed value up to the end of the fold's train span, trailing NaN included, truncated to the last `context_length` observations. Predictions therefore always start at the beginning of the test window, even if the series has NaN at the end of the train span.
- A series is **predicted in a fold only if** it has at least one observed value in the fold's test window **and** its context window is not entirely NaN. A series that ended before the test window, or whose test window is entirely missing, is left out of that fold.
- A fold where none of the requested `levels` can be predicted is **skipped** with a `MissingValuesWarning` and contributes no rows. The metric of a level that is never predicted is `None`.
- The historical and future **exog** are sliced by fold dates and then aligned exactly as in `predict`; an exog that ends before the horizon is completed with NaN and reported with a `MissingValuesWarning` (use `suppress_warnings=True` to silence it).

In [ ]:
# Backtesting
# ==============================================================================
cv = TimeSeriesFold(
         steps              = 24,
         initial_train_size = "2016-07-31 23:59:00",
     )

metrics_levels, backtest_predictions = backtesting_foundation(
    forecaster            = forecaster,
    series                = series_dict,
    exog                  = exog_dict,
    cv                    = cv,
    levels                = None,
    metric                = "mean_absolute_error",
    add_aggregated_metric = True,
    suppress_warnings     = True
)
metrics_levels


Seven folds of 24 days cover the test period. Applying the rules above:

- `id_1002` ended on 2016-07-01, before the first test window, so it is never predicted and its metric is `None`.
- `id_1004` ends on 2016-08-31. It is predicted in folds 0 and 1 (test windows up to 2016-09-17) and left out afterwards.
- `id_1003` is left out of any fold whose test window falls entirely inside one of its NaN blocks, and predicted in the others with its context ending at the fold's train end, NaN included.
- `id_1000` and `id_1001` are predicted in every fold. `id_1001` starts with a 30-observation context in fold 0, which grows by 24 observations per fold up to `context_length`.

The aggregated metrics (`average`, `weighted_average`, `pooling`) skip the levels without predictions.

<div class="admonition note" name="html-admonition" style="background: rgba(0,184,212,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #00b8d4; border-color: #00b8d4; padding-left: 10px; padding-right: 10px;">

<p class="title">
    <i style="font-size: 18px; color:#00b8d4;"></i>
    <b style="color: #00b8d4;">&#9999;&#65039; Note</b>
</p>

<code>backtesting_foundation</code> forces <code>refit=True</code> and <code>fixed_train_size=False</code> so that the context of each series grows with every fold up to <code>context_length</code>. No weights are ever trained. See the <a href="../user_guides/backtesting.html">backtesting guide</a> for the fold parameters.

</div>


## Summary

| Question | Answer |
|:---------|:-------|
| Can the series have different lengths and spans? | Yes. Each series is forecast from its own context and horizon. |
| Can each series have a different subset of exog columns? | Yes. Each series is validated and forecast with its own columns. Backends that need identical columns per batch are called once per group of series sharing the same columns. |
| Can a series lack exog while others have it? | Yes. It is forecast without covariates in its own group. |
| Can a future exog column be absent from the history? | No. It raises a `ValueError` for that series. |
| Can the target contain NaN? | Yes for every current backend. The adapter table states what the backend does with them. An entirely NaN context is rejected. |
| Does grouping change the forecasts? | Not for Chronos-2, TS-ICL and TabICL (bit-identical to forecasting each series alone). For TimesFM 3.0 with covariates, mixed lengths within a group can shift the forecast of a shorter series slightly because of the backend's padding. |
| How do I know what a backend needs? | Inspect `allow_exog`, `supports_past_only_covariates`, `supports_heterogeneous_covariates` and `supports_nan_in_series` on the forecaster. |
